In [1]:
import pandas as pd

In [2]:
latest_df = pd.read_csv("./pipeline_steps/input_files/2021-01-2024-06-overdoses.csv")

/tmp/ipykernel_365052/136032697.py:1: DtypeWarning: Columns (7,41,42) have mixed types. Specify dtype option on import or set low_memory=False.
  latest_df = pd.read_csv("./pipeline_steps/input_files/2021-01-2024-06-overdoses.csv")


In [3]:
latest_df.to_csv("./")

IsADirectoryError: [Errno 21] Is a directory: './'

In [ ]:
latest_df.drop(columns=["DeathTime"], inplace=True)

In [ ]:
duplicates = latest_df[latest_df.duplicated(subset="CaseNumber", keep=False)]
print(duplicates.head())  # Displays the first few duplicate rows

   CaseNumber       Age  Gender             Race       ResidenceType  \
0  2024-01956  37 Years    Male  Hispanic/Latino           Residence   
1  2024-01862  54 Years    Male  Hispanic/Latino                 NaN   
2  2024-01764  47 Years    Male  Hispanic/Latino  Private Residence    
3  2024-01771  49 Years  Female  Hispanic/Latino           Residence   
4  2024-01673  73 Years    Male  Hispanic/Latino          Residence`   

   ExperiencingHomelessness   DeathDate          DeathPlace      DeathCity  \
0                       NaN  2024-01-31           Residence        Burbank   
1                       NaN  2024-01-29  Private residence        Monrovia   
2                       NaN  2024-01-27  Private Residence     Pico Rivera   
3                       NaN  2024-01-27           Residence  Panorama City   
4                       NaN  2024-01-26          Residence`    Los Angeles   

  DeathZip  ...    LABEL   ShapeSTArea ShapeSTLength OBJECTID_right  ZIPCODE  \
0    91504  ...  3

In [ ]:
duplicates_sorted = duplicates.sort_values(by="CaseNumber")

In [ ]:
grouped = duplicates_sorted.groupby("CaseNumber")


# Step 4: Define a function to compare rows within each group
def find_differences(group):
    # If the group has only one row, no differences to find
    if len(group) == 1:
        return pd.DataFrame()
    else:
        # Initialize an empty DataFrame to store differences
        differences = pd.DataFrame()
        # Get all column names except 'CaseNumber'
        columns = group.columns.drop("CaseNumber")
        # Compare each column
        for column in columns:
            # Check if all values in the column are the same
            if group[column].nunique() > 1:
                # If not, include this column in differences
                differences[column] = group[column]
        # Add 'CaseNumber' to keep track of the group
        differences["CaseNumber"] = group["CaseNumber"]
        return differences


# Step 5: Apply the function to each group and collect the results
differences_list = []
for name, group in grouped:
    diff = find_differences(group)
    if not diff.empty:
        differences_list.append(diff)

# Concatenate all differences into a single DataFrame
differences_df = pd.concat(differences_list)

In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
differences_df.to_csv("./pipeline_differences.csv")

In [27]:
# Assume df is your DataFrame
# List of columns to check
columns_to_check = [
    "Methamphetamine",
    "Heroin",
    "Cocaine",
    "Fentanyl",
    "Alcohol",
    "Prescription.opioids",
    "Any Opioids",
    "Benzodiazepines",
    "Others",
    "Any Drugs",
    "Drug No Opioids",
    "EventAddress",
]


# Function to resolve duplicates based on your criteria
def resolve_duplicates(group):
    # If there's only one row, return it as is
    if len(group) == 1:
        return group

    # Step 1: Keep rows where the specified columns have '1'
    for col in columns_to_check[:-1]:  # Exclude 'EventAddress' for now
        max_value = group[col].max()
        group = group[group[col] == max_value]
        if len(group) == 1:
            return group

    # Step 2: If the values are the same, compare 'EventAddress' length
    group["EventAddress_length"] = group["EventAddress"].astype(str).str.len()
    max_length = group["EventAddress_length"].max()
    group = group[group["EventAddress_length"] == max_length]
    group = group.drop(columns="EventAddress_length")

    # Step 3: If still multiple rows, keep the first one (it doesn't matter which)
    return group.iloc[[0]]


# Apply the function to each group of duplicates
processed_df = latest_df.groupby("CaseNumber", group_keys=False).apply(
    resolve_duplicates
)

# Reset index if needed
processed_df = processed_df.reset_index(drop=True)

# Display the resulting DataFrame
print(processed_df)

       CaseNumber   Age Gender                     Race ResidenceType  \
0      2012-00007  56.0   Male  HISPANIC/LATIN AMERICAN           NaN   
1      2012-00017  51.0   Male                 FILIPINO           NaN   
2      2012-00018  50.0   Male  HISPANIC/LATIN AMERICAN           NaN   
3      2012-00071  47.0   Male  HISPANIC/LATIN AMERICAN          HOME   
4      2012-00097  29.0   Male                    BLACK          HOME   
...           ...   ...    ...                      ...           ...   
19759  2024-10450   NaN      M            Unknown/Other           NaN   
19760  2024-10451   NaN      M                      NaN           NaN   
19761  2024-10452   NaN      M          Hispanic/Latino           NaN   
19762  2024-10453   NaN      F          White/Caucasian           NaN   
19763  2024-10457   NaN      F          White/Caucasian           NaN   

       ExperiencingHomelessness   DeathDate            DeathPlace  \
0                           NaN         NaN           

/tmp/ipykernel_353430/4241692643.py:43: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  processed_df = latest_df.groupby("CaseNumber", group_keys=False).apply(


In [31]:
processed_df.to_csv("2021-01-2024-06-ods.csv")